In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU count:", torch.cuda.device_count())
    print("GPU memory (GB):",
          round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
else:
    print("No GPU detected.")

In [ ]:
import os

INPUT_DIR = "/kaggle/input"

print("Kaggle input datasets:")

for item in os.listdir(INPUT_DIR):
    print("\nDataset:", item)

    dataset_path = os.path.join(INPUT_DIR, item)

    for root, dirs, files in os.walk(dataset_path):
        level = root.replace(dataset_path, "").count(os.sep)

        if level > 2:
            dirs[:] = []
            continue

        print("  " * level + os.path.basename(root) + "/")

        for file in files[:3]:
            print("  " * (level + 1) + file)

In [ ]:
import os

DATASET_PATH = "/kaggle/input/datasets/uraninjo/augmented-alzheimer-mri-dataset"

print("Dataset exists:", os.path.exists(DATASET_PATH))
print("\nFolder structure:\n")

for root, dirs, files in os.walk(DATASET_PATH):
    level = root.replace(DATASET_PATH, "").count(os.sep)

    if level > 2:
        dirs[:] = []
        continue

    print("  " * level + os.path.basename(root) + "/")

    for folder in sorted(dirs):
        folder_path = os.path.join(root, folder)

        image_count = sum(
            1 for f in os.listdir(folder_path)
            if f.lower().endswith((".jpg", ".jpeg", ".png"))
        )

        print("  " * (level + 1) + f"{folder}/ ({image_count} images)")

In [ ]:
import os
import pandas as pd

INPUT_DIR = "/kaggle/input"

# Find the uploaded manifest
manifest_files = []

for root, dirs, files in os.walk(INPUT_DIR):
    for file in files:
        if file.lower() == "dataset_manifest.csv":
            manifest_files.append(os.path.join(root, file))

print("Manifest files found:", len(manifest_files))

for path in manifest_files:
    print(path)

if len(manifest_files) != 1:
    raise RuntimeError(
        "Expected exactly one dataset_manifest.csv. "
        "Check the attached Kaggle inputs."
    )

MANIFEST_PATH = manifest_files[0]

df = pd.read_csv(MANIFEST_PATH)

print("\nManifest columns:")
print(df.columns.tolist())

print("\nTotal records:", len(df))

print("\nSplit distribution:")
print(df["split"].value_counts())

print("\nClass distribution:")
print(pd.crosstab(df["split"], df["class"]))

print("\nSample image paths:")
print(df["path"].head(5).to_string(index=False))

In [ ]:

import os
import pandas as pd
from pathlib import Path

# Kaggle dataset directory
DATASET_PATH = (
    "/kaggle/input/datasets/uraninjo/"
    "augmented-alzheimer-mri-dataset"
)

# Load manifest
df = pd.read_csv(MANIFEST_PATH)

def resolve_kaggle_path(row):

    # Convert Windows separators to Linux-compatible paths
    relative_path = row["path"].replace("\\", "/")

    # Remove the original train/val prefix
    parts = relative_path.split("/")

    if len(parts) != 3:
        raise ValueError(
            f"Unexpected image path: {row['path']}"
        )

    original_split = str(row["original_split"]).lower()

    if original_split == "train":
        source_folder = "AugmentedAlzheimerDataset"

    elif original_split == "val":
        source_folder = "OriginalDataset"

    else:
        raise ValueError(
            f"Unknown original split: {original_split}"
        )

    return os.path.join(
        DATASET_PATH,
        source_folder,
        parts[1],
        parts[2]
    )


# Create Kaggle-compatible image paths
df["kaggle_path"] = df.apply(
    resolve_kaggle_path,
    axis=1
)

# Check whether files exist
df["file_exists"] = df["kaggle_path"].apply(
    os.path.isfile
)

print("TOTAL IMAGES:", len(df))
print("EXISTING IMAGES:", df["file_exists"].sum())
print("MISSING IMAGES:", (~df["file_exists"]).sum())

print("\nOriginal source distribution:")
print(df["original_split"].value_counts())

print("\nNew experimental split:")
print(df["split"].value_counts())

print("\nFile verification by split:")
print(
    df.groupby("split")["file_exists"]
    .agg(["count", "sum"])
)

print("\nSample mapped paths:")
print(
    df[["path", "kaggle_path", "file_exists"]]
    .head(5)
    .to_string(index=False)
)

if not df["file_exists"].all():
    print("\nWARNING: Some image files are missing.")
else:
    print("\nSUCCESS: All manifest paths resolve to existing images.")


In [ ]:
import pandas as pd

# Check exact duplicate leakage using SHA-256
duplicate_leakage = (
    df.groupby("sha256")["split"]
    .nunique()
)

leaking_groups = duplicate_leakage[
    duplicate_leakage > 1
]

print("EXACT DUPLICATE LEAKAGE AUDIT")
print("=============================")

print("Unique SHA-256 hashes:", df["sha256"].nunique())

print(
    "Duplicate groups crossing splits:",
    len(leaking_groups)
)

print("\nOriginal source distribution by experimental split:")

print(
    pd.crosstab(
        df["split"],
        df["original_split"],
        margins=True
    )
)

print("\nDuplicate group size distribution:")

print(
    df["duplicate_group_size"]
    .value_counts()
    .sort_index()
    .head(10)
)

if len(leaking_groups) == 0:
    print("\nPASS: No exact duplicate groups cross splits.")
else:
    print("\nWARNING: Exact duplicate leakage detected.")

In [ ]:
import os
import pandas as pd

# Locate the uploaded CSV
candidate_files = []

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file == "near_duplicate_candidates.csv":
            candidate_files.append(os.path.join(root, file))

print("Files found:", len(candidate_files))

if len(candidate_files) != 1:
    raise RuntimeError(
        "Expected exactly one near_duplicate_candidates.csv"
    )

NEAR_DUPLICATE_PATH = candidate_files[0]

# Load CSV
near_df = pd.read_csv(NEAR_DUPLICATE_PATH)

print("\nTotal candidate pairs:", len(near_df))

print("\nCSV columns:")
print(near_df.columns.tolist())

print("\nFirst five records:")
print(near_df.head().to_string(index=False))

print("\nMissing values:")
print(near_df.isnull().sum())

print("\nDistance distribution:")
if "distance" in near_df.columns:
    print(near_df["distance"].value_counts().sort_index())

In [ ]:
import pandas as pd

print("NEAR-DUPLICATE ANALYSIS")
print("=======================")

# 1. Overall perceptual hash distance distribution
print("\n1. PHASH DISTANCE DISTRIBUTION")

print(
    near_df["phash_distance"]
    .value_counts()
    .sort_index()
)

# 2. Pair distribution by split combination
print("\n2. SPLIT COMBINATIONS")

print(
    near_df.groupby(
        ["split_1", "split_2"]
    ).size().sort_values(ascending=False)
)

# 3. Cross-split candidate pairs
cross_split = near_df[
    near_df["split_1"] != near_df["split_2"]
].copy()

print("\n3. CROSS-SPLIT CANDIDATES")

print("Total cross-split pairs:", len(cross_split))

print("\nCross-split distance distribution:")

print(
    cross_split["phash_distance"]
    .value_counts()
    .sort_index()
)

# 4. Class-wise cross-split candidates
print("\n4. CLASS-WISE CROSS-SPLIT CANDIDATES")

print(
    cross_split.groupby("class")
    .size()
    .sort_values(ascending=False)
)

# 5. Cross-split candidates by distance and class
print("\n5. CLASS VS DISTANCE")

print(
    pd.crosstab(
        cross_split["class"],
        cross_split["phash_distance"]
    )
)

# 6. How many unique image paths are involved?
unique_paths = set(cross_split["path_1"]) | set(cross_split["path_2"])

print("\n6. UNIQUE IMAGES INVOLVED")

print("Unique image paths in cross-split pairs:", len(unique_paths))

In [ ]:

# Map every image path to its NEW experimental split
split_map = dict(zip(df["path"], df["split"]))

# Map candidate image paths using the manifest
near_df["experimental_split_1"] = near_df["path_1"].map(split_map)
near_df["experimental_split_2"] = near_df["path_2"].map(split_map)

# Check missing mappings
missing = near_df[
    near_df["experimental_split_1"].isna() |
    near_df["experimental_split_2"].isna()
]

print("ACTUAL EXPERIMENTAL SPLIT AUDIT")
print("===============================")

print("Total candidate pairs:", len(near_df))
print("Pairs with missing manifest paths:", len(missing))

if len(missing) > 0:
    print("\nExample missing paths:")
    print(missing[["path_1", "path_2"]].head())
    raise ValueError("Some candidate paths are missing from the manifest.")

# Identify candidate pairs crossing NEW experimental splits
actual_cross = near_df[
    near_df["experimental_split_1"] !=
    near_df["experimental_split_2"]
].copy()

print("\nCandidate pairs crossing experimental splits:")
print(len(actual_cross))

print("\nSplit combinations:")
print(
    actual_cross.groupby(
        ["experimental_split_1", "experimental_split_2"]
    ).size()
)

print("\nDistance distribution:")
print(
    actual_cross["phash_distance"]
    .value_counts()
    .sort_index()
)

print("\nClass-wise candidate pairs:")
print(
    actual_cross.groupby("class")
    .size()
    .sort_values(ascending=False)
)

print("\nUnique images involved:")
print(
    len(
        set(actual_cross["path_1"]) |
        set(actual_cross["path_2"])
    )
)

print("\nAudit completed.")


In [ ]:

import matplotlib.pyplot as plt
from PIL import Image
import random
import os

# Create lookup from original manifest path to Kaggle path
path_map = dict(zip(df["path"], df["kaggle_path"]))

random.seed(42)

classes = [
    "MildDemented",
    "ModerateDemented",
    "NonDemented",
    "VeryMildDemented"
]

distances = [0, 2, 4]

# Select up to 2 pairs per class per distance
sampled_groups = []

for class_name in classes:

    for distance in distances:

        subset = actual_cross[
            (actual_cross["class"] == class_name) &
            (actual_cross["phash_distance"] == distance)
        ]

        if len(subset) == 0:
            continue

        sampled = subset.sample(
            n=min(2, len(subset)),
            random_state=42
        )

        for _, row in sampled.iterrows():
            sampled_groups.append(row)

print("Selected image pairs:", len(sampled_groups))

# Display pairs
fig, axes = plt.subplots(
    len(sampled_groups),
    2,
    figsize=(10, 4 * len(sampled_groups))
)

if len(sampled_groups) == 1:
    axes = [axes]

for i, row in enumerate(sampled_groups):

    path1 = path_map[row["path_1"]]
    path2 = path_map[row["path_2"]]

    img1 = Image.open(path1).convert("RGB")
    img2 = Image.open(path2).convert("RGB")

    axes[i][0].imshow(img1, cmap="gray")
    axes[i][0].set_title(
        f"Image 1 | {row['class']} | "
        f"{row['experimental_split_1']}"
    )

    axes[i][1].imshow(img2, cmap="gray")
    axes[i][1].set_title(
        f"Image 2 | {row['class']} | "
        f"{row['experimental_split_2']}"
    )

    axes[i][0].axis("off")
    axes[i][1].axis("off")

    print(
        f"Pair {i+1}: "
        f"Class={row['class']}, "
        f"Distance={row['phash_distance']}, "
        f"Splits={row['experimental_split_1']} vs "
        f"{row['experimental_split_2']}"
    )

plt.tight_layout()
plt.show()


In [ ]:
!pip install scikit-image

In [ ]:

import numpy as np
import pandas as pd
from PIL import Image
from skimage.metrics import structural_similarity
import random

random.seed(42)

# Select 5 pairs for every class and pHash distance
audit_samples = []

for class_name in classes:
    for distance in [0, 2, 4]:

        subset = actual_cross[
            (actual_cross["class"] == class_name) &
            (actual_cross["phash_distance"] == distance)
        ]

        if len(subset) > 0:
            selected = subset.sample(
                n=min(5, len(subset)),
                random_state=42
            )

            audit_samples.extend(
                [row for _, row in selected.iterrows()]
            )


def calculate_ssim(path1, path2):

    image1 = Image.open(path1).convert("L").resize((128, 128))
    image2 = Image.open(path2).convert("L").resize((128, 128))

    array1 = np.asarray(image1)
    array2 = np.asarray(image2)

    return structural_similarity(
        array1,
        array2,
        data_range=255
    )


results = []

for row in audit_samples:

    image1 = path_map[row["path_1"]]
    image2 = path_map[row["path_2"]]

    score = calculate_ssim(image1, image2)

    results.append({
        "class": row["class"],
        "phash_distance": row["phash_distance"],
        "split_1": row["experimental_split_1"],
        "split_2": row["experimental_split_2"],
        "ssim": score,
        "path_1": row["path_1"],
        "path_2": row["path_2"]
    })

ssim_df = pd.DataFrame(results)

print("SSIM AUDIT")
print("===========")

print("Pairs evaluated:", len(ssim_df))

print("\nSSIM statistics by pHash distance:")

print(
    ssim_df.groupby("phash_distance")["ssim"]
    .agg(["count", "mean", "median", "min", "max"])
)

print("\nIndividual results:")

print(
    ssim_df[
        ["class", "phash_distance", "ssim"]
    ].to_string(index=False)
)

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

selected_pairs = []

for distance in [0, 2, 4]:

    subset = ssim_df[
        ssim_df["phash_distance"] == distance
    ]

    # Select lowest and highest SSIM pair
    selected_pairs.append(subset.loc[subset["ssim"].idxmin()])
    selected_pairs.append(subset.loc[subset["ssim"].idxmax()])

fig, axes = plt.subplots(
    len(selected_pairs), 2,
    figsize=(10, 24)
)

for i, row in enumerate(selected_pairs):

    image1 = Image.open(
        path_map[row["path_1"]]
    ).convert("RGB")

    image2 = Image.open(
        path_map[row["path_2"]]
    ).convert("RGB")

    axes[i, 0].imshow(image1)
    axes[i, 0].set_title(
        f"Image 1 | pHash {row['phash_distance']}"
    )

    axes[i, 1].imshow(image2)
    axes[i, 1].set_title(
        f"Image 2 | SSIM {row['ssim']:.3f}"
    )

    axes[i, 0].axis("off")
    axes[i, 1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd

# Select only images from the original dataset
original_df = df[
    df["original_split"] == "val"
].copy().reset_index(drop=True)

print("ORIGINAL-ONLY DATASET")
print("=====================")

print("Total images:", len(original_df))

print("\nClass distribution:")
print(original_df["class"].value_counts().sort_index())

print("\nExact duplicate audit:")

print(
    "Unique SHA-256 hashes:",
    original_df["sha256"].nunique()
)

print(
    "Duplicate images:",
    len(original_df) - original_df["sha256"].nunique()
)

print("\nImage path verification:")

print(
    "Existing images:",
    original_df["kaggle_path"].apply(
        lambda p: __import__("os").path.isfile(p)
    ).sum()
)

print("\nSample images:")
print(
    original_df[
        ["class", "path", "kaggle_path"]
    ].head().to_string(index=False)
)

In [ ]:

import pandas as pd

from sklearn.model_selection import train_test_split

RANDOM_SEED = 42

# First split: 70% training, 30% temporary
train_df, temp_df = train_test_split(
    original_df,
    test_size=0.30,
    random_state=RANDOM_SEED,
    stratify=original_df["class"]
)

# Second split: 15% validation, 15% testing
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=RANDOM_SEED,
    stratify=temp_df["class"]
)

# Assign split names
train_df = train_df.copy()
val_df = val_df.copy()
test_df = test_df.copy()

train_df["split"] = "train"
val_df["split"] = "val"
test_df["split"] = "test"

# Combine all splits
original_manifest = pd.concat(
    [train_df, val_df, test_df],
    ignore_index=True
)

# Shuffle manifest rows reproducibly
original_manifest = original_manifest.sample(
    frac=1,
    random_state=RANDOM_SEED
).reset_index(drop=True)

print("NEW ORIGINAL-ONLY SPLIT")
print("=======================")

print("\nTotal images:", len(original_manifest))

print("\nSplit distribution:")
print(original_manifest["split"].value_counts())

print("\nClass-wise split distribution:")
print(
    pd.crosstab(
        original_manifest["split"],
        original_manifest["class"]
    )
)

print("\nExact duplicate leakage check:")

leakage = (
    original_manifest.groupby("sha256")["split"].nunique()
)

print("Groups crossing splits:", (leakage > 1).sum())

assert len(original_manifest) == 6400
assert original_manifest["split"].notna().all()
assert (leakage > 1).sum() == 0

print("\nOriginal-only split generated successfully.")


In [ ]:
import os
import pandas as pd

OUTPUT_DIR = "/kaggle/working/acal_ad"
os.makedirs(OUTPUT_DIR, exist_ok=True)

MANIFEST_OUTPUT = os.path.join(
    OUTPUT_DIR,
    "original_only_manifest.csv"
)

# Save the finalized split
original_manifest.to_csv(
    MANIFEST_OUTPUT,
    index=False
)

# Verify saved manifest
saved_manifest = pd.read_csv(MANIFEST_OUTPUT)

print("ORIGINAL-ONLY MANIFEST")
print("======================")

print("Saved location:", MANIFEST_OUTPUT)
print("Total records:", len(saved_manifest))

print("\nSplit counts:")
print(saved_manifest["split"].value_counts())

print("\nClass distribution:")
print(
    pd.crosstab(
        saved_manifest["split"],
        saved_manifest["class"]
    )
)

print("\nMissing image paths:")
print(
    (~saved_manifest["kaggle_path"].apply(os.path.isfile)).sum()
)

print("\nManifest saved and verified.")

In [ ]:

import os
import time
import numpy as np
import pandas as pd
import imagehash

from PIL import Image
from tqdm.auto import tqdm

start_time = time.time()

# Work from the finalized original-only manifest
audit_df = pd.read_csv(
    "/kaggle/working/acal_ad/original_only_manifest.csv"
)

# 1. Calculate perceptual hashes
print("CALCULATING PERCEPTUAL HASHES")
print("=============================")

hash_values = []

for path in tqdm(audit_df["kaggle_path"]):
    with Image.open(path) as img:
        h = imagehash.phash(img.convert("RGB"))

    hash_values.append(int(str(h), 16))

audit_df["phash_int"] = hash_values

print("Images hashed:", len(audit_df))

# 2. Fast Hamming distance calculation
# Lookup table counts set bits in each byte
POPCOUNT = np.array(
    [bin(i).count("1") for i in range(256)],
    dtype=np.uint8
)

candidate_records = []

print("\nCOMPARING IMAGES WITHIN EACH CLASS")
print("==================================")

for class_name in audit_df["class"].unique():

    class_data = audit_df[
        audit_df["class"] == class_name
    ].reset_index(drop=True)

    hashes = np.array(
        class_data["phash_int"].tolist(),
        dtype=np.uint64
    )

    n = len(class_data)
    class_count = 0

    for i in tqdm(range(n), desc=class_name):

        # Compare image i only with images after it
        xor_values = np.bitwise_xor(
            hashes[i],
            hashes[i + 1:]
        )

        byte_values = xor_values.view(np.uint8).reshape(-1, 8)

        distances = POPCOUNT[byte_values].sum(axis=1)

        matching_indices = np.flatnonzero(distances <= 4)

        for offset in matching_indices:

            j = i + 1 + int(offset)

            row1 = class_data.iloc[i]
            row2 = class_data.iloc[j]

            distance = int(distances[offset])

            candidate_records.append({
                "class": class_name,
                "split_1": row1["split"],
                "split_2": row2["split"],
                "path_1": row1["path"],
                "path_2": row2["path"],
                "kaggle_path_1": row1["kaggle_path"],
                "kaggle_path_2": row2["kaggle_path"],
                "phash_distance": distance
            })

            class_count += 1

    print(class_name, "candidate pairs:", class_count)

# 3. Create candidate DataFrame
original_near_df = pd.DataFrame(candidate_records)

# 4. Save results
OUTPUT_DIR = "/kaggle/working/acal_ad"

CANDIDATE_PATH = os.path.join(
    OUTPUT_DIR,
    "original_only_near_duplicate_candidates.csv"
)

original_near_df.to_csv(
    CANDIDATE_PATH,
    index=False
)

# 5. Report results
cross_split = original_near_df[
    original_near_df["split_1"] !=
    original_near_df["split_2"]
]

print("\nORIGINAL-ONLY NEAR-DUPLICATE AUDIT")
print("=================================")

print("Total images:", len(audit_df))
print("Total candidate pairs:", len(original_near_df))

print("\nDistance distribution:")
print(
    original_near_df["phash_distance"]
    .value_counts()
    .sort_index()
)

print("\nSplit combinations:")
print(
    original_near_df.groupby(
        ["split_1", "split_2"]
    ).size()
)

print("\nCross-split candidate pairs:", len(cross_split))

print("\nCross-split distance distribution:")
print(
    cross_split["phash_distance"]
    .value_counts()
    .sort_index()
)

print("\nCandidate pairs by class:")
print(
    cross_split.groupby("class").size()
)

print("\nUnique images in cross-split candidates:")

if len(cross_split) > 0:
    print(
        len(
            set(cross_split["path_1"]) |
            set(cross_split["path_2"])
        )
    )
else:
    print(0)

print("\nSaved:", CANDIDATE_PATH)
print("Elapsed minutes:", round((time.time() - start_time) / 60, 2))


In [ ]:

import os
import numpy as np
import pandas as pd

from PIL import Image
from tqdm.auto import tqdm
from skimage.metrics import structural_similarity

# Load original-only candidate pairs
candidates = pd.read_csv(
    "/kaggle/working/acal_ad/"
    "original_only_near_duplicate_candidates.csv"
)

# Select only cross-split pairs
cross_candidates = candidates[
    candidates["split_1"] != candidates["split_2"]
].copy().reset_index(drop=True)


def calculate_ssim(path1, path2):

    with Image.open(path1) as img1:
        image1 = np.asarray(
            img1.convert("L").resize((128, 128))
        )

    with Image.open(path2) as img2:
        image2 = np.asarray(
            img2.convert("L").resize((128, 128))
        )

    return structural_similarity(
        image1,
        image2,
        data_range=255
    )


ssim_scores = []

print("Calculating SSIM for cross-split candidates...")

for _, row in tqdm(cross_candidates.iterrows(),
                   total=len(cross_candidates)):

    score = calculate_ssim(
        row["kaggle_path_1"],
        row["kaggle_path_2"]
    )

    ssim_scores.append(score)


cross_candidates["ssim"] = ssim_scores

# Save results
OUTPUT_PATH = (
    "/kaggle/working/acal_ad/"
    "original_only_ssim_audit.csv"
)

cross_candidates.to_csv(
    OUTPUT_PATH,
    index=False
)

print("\nSSIM AUDIT SUMMARY")
print("==================")

print("Pairs evaluated:", len(cross_candidates))

print("\nSSIM statistics:")

print(
    cross_candidates["ssim"].describe()
)

print("\nSSIM by pHash distance:")

print(
    cross_candidates.groupby("phash_distance")["ssim"]
    .agg(["count", "mean", "median", "min", "max"])
)

print("\nHigh similarity candidate counts:")

for threshold in [0.80, 0.90, 0.95, 0.98]:
    count = (
        cross_candidates["ssim"] >= threshold
    ).sum()

    print(f"SSIM >= {threshold}: {count}")

print("\nSaved:", OUTPUT_PATH)


In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

ssim_file = (
    "/kaggle/working/acal_ad/"
    "original_only_ssim_audit.csv"
)

ssim_results = pd.read_csv(ssim_file)

review_pairs = []

for class_name in classes:

    subset = ssim_results[
        ssim_results["class"] == class_name
    ]

    # Highest similarity examples
    high = subset[subset["ssim"] >= 0.95].nlargest(
        4, "ssim"
    )

    # High but below 0.95
    medium_high = subset[
        (subset["ssim"] >= 0.90) &
        (subset["ssim"] < 0.95)
    ].nlargest(2, "ssim")

    # Moderate similarity examples
    medium = subset[
        (subset["ssim"] >= 0.80) &
        (subset["ssim"] < 0.90)
    ].nlargest(1, "ssim")

    review_pairs.extend([
        row for _, row in pd.concat(
            [high, medium_high, medium]
        ).iterrows()
    ])

print("Selected pairs:", len(review_pairs))

fig, axes = plt.subplots(
    len(review_pairs), 2,
    figsize=(10, max(5, 3.5 * len(review_pairs)))
)

if len(review_pairs) == 1:
    axes = [axes]

for i, row in enumerate(review_pairs):

    image1 = Image.open(
        row["kaggle_path_1"]
    ).convert("RGB")

    image2 = Image.open(
        row["kaggle_path_2"]
    ).convert("RGB")

    axes[i][0].imshow(image1)
    axes[i][0].set_title(
        f"{row['class']} | {row['split_1']}"
    )

    axes[i][1].imshow(image2)
    axes[i][1].set_title(
        f"SSIM: {row['ssim']:.3f} | "
        f"{row['split_2']} | "
        f"pHash: {row['phash_distance']}"
    )

    axes[i][0].axis("off")
    axes[i][1].axis("off")

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np

# Load the existing SSIM audit if needed
if "ssim_results" not in globals():
    ssim_results = pd.read_csv(
        "/kaggle/working/acal_ad/original_only_ssim_audit.csv"
    )

# Automatic screening categories
def classify_pair(row):
    phash = row["phash_distance"]
    ssim = row["ssim"]

    if phash == 0 and ssim >= 0.90:
        return "Likely duplicate - inspect"
    elif phash <= 2 and ssim >= 0.90:
        return "Possible modified copy - inspect"
    else:
        return "Similar or uncertain - keep separate"

ssim_results["automatic_review"] = ssim_results.apply(
    classify_pair, axis=1
)

print("AUTOMATIC REVIEW SUMMARY")
print(ssim_results["automatic_review"].value_counts())

# Save results
output_path = (
    "/kaggle/working/acal_ad/automatic_duplicate_review.csv"
)
ssim_results.to_csv(output_path, index=False)

print("\nSaved:", output_path)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import math
import os

review_df = ssim_results[
    (ssim_results["phash_distance"] == 0) &
    (ssim_results["ssim"] >= 0.90)
].copy().reset_index(drop=True)

print("Pairs selected:", len(review_df))

# Display up to 12 pairs per page
pairs_per_page = 12

for start in range(0, len(review_df), pairs_per_page):
    batch = review_df.iloc[start:start + pairs_per_page]

    fig, axes = plt.subplots(
        len(batch), 2, figsize=(8, 3 * len(batch))
    )

    if len(batch) == 1:
        axes = [axes]

    for row_idx, (_, pair) in enumerate(batch.iterrows()):
        img1 = Image.open(pair["kaggle_path_1"]).convert("RGB")
        img2 = Image.open(pair["kaggle_path_2"]).convert("RGB")

        axes[row_idx][0].imshow(img1)
        axes[row_idx][0].set_title("Image 1")
        axes[row_idx][1].imshow(img2)
        axes[row_idx][1].set_title("Image 2")

        axes[row_idx][0].axis("off")
        axes[row_idx][1].axis("off")

        fig.text(
            0.5,
            1 - (row_idx + 0.5) / len(batch),
            f"Pair {start + row_idx + 1} | "
            f"SSIM: {pair['ssim']:.3f}",
            ha="center",
            fontsize=9
        )

    plt.tight_layout()
    plt.show()

In [ ]:
print("Available columns:")
print(original_manifest.columns.tolist())

In [ ]:
import os
import shutil
import pandas as pd

OUTPUT_DIR = "/kaggle/working/acal_ad"
FINAL_DIR = os.path.join(OUTPUT_DIR, "final_dataset_package")

os.makedirs(FINAL_DIR, exist_ok=True)

# Detect the class column
class_col = next(
    (c for c in ["class", "Class", "label", "Label", "category"]
     if c in original_manifest.columns),
    None
)

if class_col is None:
    raise ValueError(
        "Class column not found. Available columns: "
        + str(original_manifest.columns.tolist())
    )

# Save manifest
original_manifest.to_csv(
    os.path.join(FINAL_DIR, "final_manifest.csv"),
    index=False
)

# Save audit files
original_near_df.to_csv(
    os.path.join(FINAL_DIR, "near_duplicate_candidates.csv"),
    index=False
)

ssim_results.to_csv(
    os.path.join(FINAL_DIR, "ssim_audit.csv"),
    index=False
)

# Save split summary
split_summary = (
    original_manifest
    .groupby(["split", class_col])
    .size()
    .unstack(fill_value=0)
)

split_summary.to_csv(
    os.path.join(FINAL_DIR, "split_summary.csv")
)

# Create ZIP
zip_path = shutil.make_archive(
    "/kaggle/working/ACAL_AD_Final_Dataset_Package",
    "zip",
    FINAL_DIR
)

print("\nBACKUP CREATED SUCCESSFULLY")
print("Class column:", class_col)
print("Total images:", len(original_manifest))
print("Split summary:")
print(split_summary)
print("\nZIP:", zip_path)
print("\nFiles:", os.listdir(FINAL_DIR))

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU count:", torch.cuda.device_count())

    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}:", torch.cuda.get_device_name(i))
        print(
            "GPU memory:",
            round(
                torch.cuda.get_device_properties(i).total_memory
                / (1024**3), 2
            ),
            "GB"
        )
else:
    print("WARNING: GPU is not enabled.")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models

# Use one GPU initially
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

NUM_CLASSES = 4
NUM_BRANCHES = 3


# --------------------------------------------------
# 1. EAGLE ATTENTION
# --------------------------------------------------

class EagleAttention(nn.Module):
    def __init__(self, channels=2048):
        super().__init__()

        reduced_channels = channels // 8

        self.query = nn.Conv2d(
            channels, reduced_channels, kernel_size=1
        )

        self.key = nn.Conv2d(
            channels, reduced_channels, kernel_size=1
        )

        self.value = nn.Conv2d(
            channels, channels, kernel_size=1
        )

        self.gamma = nn.Parameter(torch.zeros(1))

    def forward(self, x):

        batch, channels, height, width = x.shape

        q = self.query(x).flatten(2).transpose(1, 2)
        k = self.key(x).flatten(2)
        v = self.value(x).flatten(2).transpose(1, 2)

        attention = torch.bmm(q, k)

        attention = attention / (q.shape[-1] ** 0.5)

        attention = F.softmax(attention, dim=-1)

        attended = torch.bmm(attention, v)

        attended = attended.transpose(1, 2).reshape(
            batch, channels, height, width
        )

        output = x + self.gamma * attended

        return output, attention


# --------------------------------------------------
# 2. REGION IMPORTANCE
# --------------------------------------------------

class RegionImportance(nn.Module):
    def __init__(self, channels=2048):
        super().__init__()

        self.region_conv = nn.Conv2d(
            channels, 1, kernel_size=1
        )

    def forward(self, x):

        region_map = torch.sigmoid(
            self.region_conv(x)
        )

        weighted_features = x * region_map

        return weighted_features, region_map


# --------------------------------------------------
# 3. WOLF PACK FUSION
# --------------------------------------------------

class WolfPackFusion(nn.Module):
    def __init__(
        self,
        channels=2048,
        num_classes=4
    ):
        super().__init__()

        self.branches = nn.ModuleList([
            nn.Linear(channels, num_classes)
            for _ in range(NUM_BRANCHES)
        ])

    def forward(self, x):

        features = F.adaptive_avg_pool2d(
            x, (1, 1)
        ).flatten(1)

        branch_logits = [
            branch(features)
            for branch in self.branches
        ]

        logits = torch.stack(
            branch_logits, dim=0
        ).mean(dim=0)

        return logits


# --------------------------------------------------
# 4. COMPLETE PREDATORNET
# --------------------------------------------------

class PredatorNet(nn.Module):
    def __init__(
        self,
        num_classes=4,
        pretrained=False
    ):
        super().__init__()

        weights = (
            models.ResNet50_Weights.DEFAULT
            if pretrained else None
        )

        resnet = models.resnet50(weights=weights)

        # Remove average pooling and final FC layer
        self.backbone = nn.Sequential(
            *list(resnet.children())[:-2]
        )

        self.eagle_attention = EagleAttention(2048)

        self.region_importance = RegionImportance(2048)

        self.wolfpack_fusion = WolfPackFusion(
            channels=2048,
            num_classes=num_classes
        )

    def forward(self, x):

        features = self.backbone(x)

        attended_features, attention_map = (
            self.eagle_attention(features)
        )

        weighted_features, region_map = (
            self.region_importance(attended_features)
        )

        logits = self.wolfpack_fusion(weighted_features)

        return {
            "logits": logits,
            "region_map": region_map,
            "attention_map": attention_map
        }


print("PredatorNet architecture classes created.")
print("Device:", DEVICE)

In [ ]:
# Create model without pretrained weights for shape testing
model = PredatorNet(
    num_classes=NUM_CLASSES,
    pretrained=False
).to(DEVICE)

# Small dummy batch
dummy_input = torch.randn(
    2, 3, 224, 224
).to(DEVICE)

model.eval()

with torch.no_grad():
    output = model(dummy_input)

print("INPUT SHAPE:", dummy_input.shape)

print("LOGITS SHAPE:", output["logits"].shape)

print("REGION MAP SHAPE:", output["region_map"].shape)

print("ATTENTION MAP SHAPE:", output["attention_map"].shape)

total_params = sum(
    p.numel() for p in model.parameters()
)

trainable_params = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

print("\nTOTAL PARAMETERS:", f"{total_params:,}")
print("TRAINABLE PARAMETERS:", f"{trainable_params:,}")

print("\nPREDATORNET ARCHITECTURE TEST COMPLETED")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class BaselineBGRALoss(nn.Module):

    def __init__(self, lambda_smooth=0.01):
        super().__init__()

        self.lambda_smooth = lambda_smooth
        self.ce_loss = nn.CrossEntropyLoss()

    def forward(self, outputs, targets):

        logits = outputs["logits"]
        region_map = outputs["region_map"]

        # 1. Classification loss
        classification_loss = self.ce_loss(
            logits, targets
        )

        # 2. Horizontal smoothness
        horizontal_diff = torch.abs(
            region_map[:, :, :, 1:] -
            region_map[:, :, :, :-1]
        )

        # 3. Vertical smoothness
        vertical_diff = torch.abs(
            region_map[:, :, 1:, :] -
            region_map[:, :, :-1, :]
        )

        smoothness_loss = (
            horizontal_diff.mean() +
            vertical_diff.mean()
        )

        # 4. Total BGRA loss
        total_loss = (
            classification_loss +
            self.lambda_smooth * smoothness_loss
        )

        return {
            "total_loss": total_loss,
            "classification_loss": classification_loss,
            "smoothness_loss": smoothness_loss
        }


# Test baseline loss
criterion = BaselineBGRALoss(
    lambda_smooth=0.01
).to(DEVICE)

dummy_targets = torch.tensor(
    [0, 2],
    dtype=torch.long,
    device=DEVICE
)

model.train()

dummy_outputs = model(dummy_input)

loss_results = criterion(
    dummy_outputs,
    dummy_targets
)

print("BASELINE BGRA LOSS TEST")
print("-----------------------")
print("Total loss:",
      loss_results["total_loss"].item())

print("Classification loss:",
      loss_results["classification_loss"].item())

print("Smoothness loss:",
      loss_results["smoothness_loss"].item())

print("\nBGRA LOSS TEST COMPLETED")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class AdaptiveConfusionAwareLoss(nn.Module):

    def __init__(
        self,
        lambda_smooth=0.01,
        lambda_confusion=0.10,
        num_classes=4,
        ema_momentum=0.90,
        smoothing=1.0
    ):
        super().__init__()

        self.lambda_smooth = lambda_smooth
        self.lambda_confusion = lambda_confusion
        self.num_classes = num_classes
        self.ema_momentum = ema_momentum
        self.smoothing = smoothing

        self.ce_loss = nn.CrossEntropyLoss()

        # Initially, all incorrect classes receive equal weight.
        initial_weights = torch.ones(
            num_classes, num_classes
        ) - torch.eye(num_classes)

        initial_weights = initial_weights / (num_classes - 1)

        self.register_buffer(
            "confusion_weights", initial_weights
        )

        # Counts accumulated only from training predictions.
        self.register_buffer(
            "epoch_confusion_counts",
            torch.zeros(num_classes, num_classes)
        )

    def update_training_statistics(self, logits, targets):
        """
        Call only with training batches.
        Uses detached predictions; no gradients flow through statistics.
        """

        with torch.no_grad():

            predictions = logits.detach().argmax(dim=1)

            indices = (
                targets.detach() * self.num_classes
                + predictions
            )

            batch_counts = torch.bincount(
                indices,
                minlength=self.num_classes ** 2
            ).reshape(
                self.num_classes,
                self.num_classes
            )

            self.epoch_confusion_counts.add_(
                batch_counts.to(
                    self.epoch_confusion_counts.device,
                    dtype=self.epoch_confusion_counts.dtype
                )
            )

    def finish_training_epoch(self):
        """
        Update adaptive confusion weights after a full training epoch.
        Do not call using validation or test predictions.
        """

        with torch.no_grad():

            counts = self.epoch_confusion_counts.clone()

            # Ignore correct predictions.
            counts.fill_diagonal_(0)

            # Smooth the off-diagonal estimates.
            counts = counts + self.smoothing * (
                1 - torch.eye(
                    self.num_classes,
                    device=counts.device
                )
            )

            row_sums = counts.sum(dim=1, keepdim=True)

            new_weights = counts / row_sums.clamp_min(1e-12)

            # Exponential moving average.
            updated_weights = (
                self.ema_momentum * self.confusion_weights
                + (1 - self.ema_momentum) * new_weights
            )

            updated_weights.fill_diagonal_(0)

            updated_weights = (
                updated_weights /
                updated_weights.sum(dim=1, keepdim=True).clamp_min(1e-12)
            )

            self.confusion_weights.copy_(updated_weights)

            self.epoch_confusion_counts.zero_()

    def forward(self, outputs, targets):

        logits = outputs["logits"]
        region_map = outputs["region_map"]

        # 1. Classification loss
        classification_loss = self.ce_loss(
            logits, targets
        )

        # 2. Region smoothness loss
        horizontal_diff = torch.abs(
            region_map[:, :, :, 1:] -
            region_map[:, :, :, :-1]
        )

        vertical_diff = torch.abs(
            region_map[:, :, 1:, :] -
            region_map[:, :, :-1, :]
        )

        smoothness_loss = (
            horizontal_diff.mean()
            + vertical_diff.mean()
        )

        # 3. Adaptive confusion penalty
        probabilities = F.softmax(logits, dim=1)

        sample_weights = self.confusion_weights[targets]

        confusion_loss = (
            probabilities * sample_weights
        ).sum(dim=1).mean()

        # 4. Total ACAL loss
        total_loss = (
            classification_loss
            + self.lambda_smooth * smoothness_loss
            + self.lambda_confusion * confusion_loss
        )

        return {
            "total_loss": total_loss,
            "classification_loss": classification_loss,
            "smoothness_loss": smoothness_loss,
            "confusion_loss": confusion_loss
        }


print("ACAL loss class created successfully.")

In [ ]:
acal_criterion = AdaptiveConfusionAwareLoss(
    lambda_smooth=0.01,
    lambda_confusion=0.10,
    num_classes=4
).to(DEVICE)

model.eval()

with torch.no_grad():
    dummy_outputs = model(dummy_input)

acal_results = acal_criterion(
    dummy_outputs,
    dummy_targets
)

print("ACAL LOSS TEST")
print("-----------------------")

for name, value in acal_results.items():
    print(f"{name}: {value.item():.6f}")

print("\nINITIAL CONFUSION WEIGHTS:")
print(acal_criterion.confusion_weights)

print("\nACAL LOSS TEST COMPLETED")

In [ ]:
import os
import pandas as pd

MANIFEST_PATH = (
    "/kaggle/working/acal_ad/final_dataset_package/final_manifest.csv"
)

manifest = pd.read_csv(MANIFEST_PATH)

print("Total images:", len(manifest))
print("\nColumns:")
print(manifest.columns.tolist())

print("\nFirst five rows:")
display(manifest.head())

print("\nSplit distribution:")
print(manifest["split"].value_counts())

print("\nClass distribution:")
print(manifest.groupby(["split", "class"]).size().unstack(fill_value=0))

print("\nChecking image paths...")

missing = [
    path for path in manifest["kaggle_path"]
    if not os.path.exists(path)
]

print("Missing images:", len(missing))

In [ ]:
import torch
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from collections import Counter

# Configuration
BATCH_SIZE = 16
NUM_WORKERS = 2
IMAGE_SIZE = 224

CLASS_NAMES = [
    "MildDemented",
    "ModerateDemented",
    "NonDemented",
    "VeryMildDemented"
]

CLASS_TO_IDX = {
    name: idx for idx, name in enumerate(CLASS_NAMES)
}

# Image preprocessing
image_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5, 0.5, 0.5],
        std=[0.5, 0.5, 0.5]
    )
])


# Dataset class
class AlzheimerDataset(Dataset):

    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        image_path = row["kaggle_path"]
        class_name = row["class"]

        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        label = CLASS_TO_IDX[class_name]

        return image, label


# Divide using the existing finalized manifest
train_df = manifest[manifest["split"] == "train"].copy()
val_df = manifest[manifest["split"] == "val"].copy()
test_df = manifest[manifest["split"] == "test"].copy()

# Create datasets
train_dataset = AlzheimerDataset(
    train_df, image_transform
)

val_dataset = AlzheimerDataset(
    val_df, image_transform
)

test_dataset = AlzheimerDataset(
    test_df, image_transform
)

# Weighted sampler for training only
train_labels = [
    CLASS_TO_IDX[name] for name in train_df["class"]
]

class_counts = Counter(train_labels)

class_weights = {
    class_id: 1.0 / class_counts[class_id]
    for class_id in range(len(CLASS_NAMES))
}

sample_weights = torch.DoubleTensor([
    class_weights[label] for label in train_labels
])

train_sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(train_dataset),
    replacement=True
)

# Create DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=train_sampler,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

print("DATASET AND DATALOADER CREATED")
print("--------------------------------")
print("Training images:", len(train_dataset))
print("Validation images:", len(val_dataset))
print("Testing images:", len(test_dataset))

print("\nNumber of batches:")
print("Train:", len(train_loader))
print("Validation:", len(val_loader))
print("Test:", len(test_loader))

# Test one batch
images, labels = next(iter(train_loader))

print("\nBATCH VERIFICATION")
print("Image shape:", images.shape)
print("Label shape:", labels.shape)
print("Image dtype:", images.dtype)
print("Label dtype:", labels.dtype)
print("Image range:", images.min().item(), images.max().item())

print("\nCLASS MAPPING:")
print(CLASS_TO_IDX)

print("\nDATALOADER TEST COMPLETED")

In [ ]:
import os
import random
import numpy as np
import torch
import json

# Reproducibility
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Experiment configuration
CONFIG = {
    "project": "ACAL-AD",
    "architecture": "PredatorNet",
    "num_classes": 4,
    "class_names": CLASS_NAMES,
    "image_size": 224,
    "batch_size": 16,
    "epochs": 30,
    "learning_rate": 0.0001,
    "weight_decay": 0.0001,
    "optimizer": "Adam",
    "seed": SEED,
    "device": "cuda:0",
    "lambda_smooth": 0.01,
    "lambda_confusion": 0.10,
    "ema_momentum": 0.90,
    "confusion_smoothing": 1.0,
    "early_stopping_patience": 7,
    "selection_metric": "validation_macro_f1"
}

# Output folders
BASE_OUTPUT = "/kaggle/working/acal_ad/training"

FOLDERS = [
    "baseline/checkpoints",
    "baseline/logs",
    "baseline/graphs",
    "acal/checkpoints",
    "acal/logs",
    "acal/graphs"
]

for folder in FOLDERS:
    os.makedirs(
        os.path.join(BASE_OUTPUT, folder),
        exist_ok=True
    )

# Save configuration
config_path = os.path.join(
    BASE_OUTPUT,
    "experiment_config.json"
)

with open(config_path, "w") as f:
    json.dump(CONFIG, f, indent=4)

print("TRAINING CONFIGURATION CREATED")
print("--------------------------------")
print(json.dumps(CONFIG, indent=4))
print("\nConfiguration saved:", config_path)
print("Output directory:", BASE_OUTPUT)

In [ ]:
from torch.utils.data import DataLoader

# Separate loader for confusion statistics.
# Uses training images only, without replacement.

train_stats_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

print("ACAL STATISTICS DATALOADER")
print("--------------------------")

print("Training images:", len(train_stats_loader.dataset))
print("Number of batches:", len(train_stats_loader))

# Verify one batch
stats_images, stats_labels = next(iter(train_stats_loader))

print("Image shape:", stats_images.shape)
print("Label shape:", stats_labels.shape)

print("\nACAL STATISTICS LOADER READY")

In [ ]:
import os
import csv
import copy
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import DataLoader, WeightedRandomSampler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# --------------------------------------------------
# 1. REPRODUCIBILITY
# --------------------------------------------------

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# --------------------------------------------------
# 2. CREATE A REPRODUCIBLE TRAINING LOADER
# --------------------------------------------------

def create_experiment_train_loader(seed=42):

    labels = [
        CLASS_TO_IDX[name] for name in train_df["class"]
    ]

    counts = Counter(labels)

    weights = [
        1.0 / counts[label] for label in labels
    ]

    generator = torch.Generator()
    generator.manual_seed(seed)

    sampler = WeightedRandomSampler(
        weights=torch.DoubleTensor(weights),
        num_samples=len(train_dataset),
        replacement=True,
        generator=generator
    )

    return DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        sampler=sampler,
        num_workers=NUM_WORKERS,
        pin_memory=True
    )


# --------------------------------------------------
# 3. EVALUATION FUNCTION
# --------------------------------------------------

def evaluate_model(model, loader, criterion, device):

    model.eval()

    total_loss = 0.0
    all_predictions = []
    all_targets = []

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with torch.amp.autocast(
                device_type="cuda",
                enabled=(device.type == "cuda")
            ):
                outputs = model(images)
                losses = criterion(outputs, labels)

            total_loss += losses["total_loss"].item()

            predictions = outputs["logits"].argmax(dim=1)

            all_predictions.extend(
                predictions.cpu().numpy()
            )

            all_targets.extend(
                labels.cpu().numpy()
            )

    accuracy = accuracy_score(
        all_targets, all_predictions
    )

    precision = precision_score(
        all_targets,
        all_predictions,
        average="macro",
        zero_division=0
    )

    recall = recall_score(
        all_targets,
        all_predictions,
        average="macro",
        zero_division=0
    )

    macro_f1 = f1_score(
        all_targets,
        all_predictions,
        average="macro",
        zero_division=0
    )

    matrix = confusion_matrix(
        all_targets,
        all_predictions,
        labels=list(range(NUM_CLASSES))
    )

    return {
        "loss": total_loss / len(loader),
        "accuracy": accuracy,
        "macro_precision": precision,
        "macro_recall": recall,
        "macro_f1": macro_f1,
        "confusion_matrix": matrix
    }


# --------------------------------------------------
# 4. ACAL TRAINING STATISTICS UPDATE
# --------------------------------------------------

def update_acal_statistics(model, criterion, loader, device):

    model.eval()

    # Reset epoch counts before collecting predictions
    criterion.epoch_confusion_counts.zero_()

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with torch.amp.autocast(
                device_type="cuda",
                enabled=(device.type == "cuda")
            ):
                outputs = model(images)

            # Training images only
            criterion.update_training_statistics(
                outputs["logits"],
                labels
            )

    # Update adaptive weights once after all training images
    criterion.finish_training_epoch()


# --------------------------------------------------
# 5. ONE TRAINING EPOCH
# --------------------------------------------------

def train_one_epoch(
    model,
    loader,
    criterion,
    optimizer,
    scaler,
    device
):

    model.train()

    running_loss = 0.0
    running_ce = 0.0
    running_smooth = 0.0
    running_confusion = 0.0

    for images, labels in loader:

        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(
            device_type="cuda",
            enabled=(device.type == "cuda")
        ):

            outputs = model(images)
            losses = criterion(outputs, labels)

            total_loss = losses["total_loss"]

        scaler.scale(total_loss).backward()

        scaler.step(optimizer)
        scaler.update()

        running_loss += total_loss.item()
        running_ce += losses["classification_loss"].item()
        running_smooth += losses["smoothness_loss"].item()

        if "confusion_loss" in losses:
            running_confusion += losses["confusion_loss"].item()

    number_of_batches = len(loader)

    return {
        "train_loss": running_loss / number_of_batches,
        "train_ce": running_ce / number_of_batches,
        "train_smooth": running_smooth / number_of_batches,
        "train_confusion": running_confusion / number_of_batches
    }


# --------------------------------------------------
# 6. COMPLETE EXPERIMENT FUNCTION
# --------------------------------------------------

def run_experiment(
    experiment_name,
    loss_type,
    initial_state=None,
    epochs=30
):

    set_seed(SEED)

    device = torch.device("cuda:0")

    print("\nStarting experiment:", experiment_name)

    # Fresh model
    model = PredatorNet(
        num_classes=NUM_CLASSES,
        pretrained=False
    ).to(device)

    if initial_state is not None:
        model.load_state_dict(copy.deepcopy(initial_state))

    # Select loss
    if loss_type == "BGRA":

        criterion = BaselineBGRALoss(
            lambda_smooth=CONFIG["lambda_smooth"]
        ).to(device)

    elif loss_type == "ACAL":

        criterion = AdaptiveConfusionAwareLoss(
            lambda_smooth=CONFIG["lambda_smooth"],
            lambda_confusion=CONFIG["lambda_confusion"],
            num_classes=NUM_CLASSES,
            ema_momentum=CONFIG["ema_momentum"],
            smoothing=CONFIG["confusion_smoothing"]
        ).to(device)

    else:
        raise ValueError("Unknown loss type")

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=CONFIG["learning_rate"],
        weight_decay=CONFIG["weight_decay"]
    )

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=(device.type == "cuda")
    )

    experiment_loader = create_experiment_train_loader(SEED)

    output_dir = os.path.join(
        BASE_OUTPUT, experiment_name
    )

    checkpoint_dir = os.path.join(
        output_dir, "checkpoints"
    )

    os.makedirs(checkpoint_dir, exist_ok=True)

    log_path = os.path.join(
        output_dir, "training_log.csv"
    )

    best_f1 = -1.0
    best_epoch = 0
    patience_counter = 0

    history = []

    # --------------------------------------------------
    # EPOCH LOOP
    # --------------------------------------------------

    for epoch in range(1, epochs + 1):

        print(
            f"\n{experiment_name} | "
            f"Epoch {epoch}/{epochs}"
        )

        train_metrics = train_one_epoch(
            model,
            experiment_loader,
            criterion,
            optimizer,
            scaler,
            device
        )

        # ACAL statistics: training images only
        if loss_type == "ACAL":

            update_acal_statistics(
                model,
                criterion,
                train_stats_loader,
                device
            )

        # Validation
        val_metrics = evaluate_model(
            model,
            val_loader,
            criterion,
            device
        )

        epoch_record = {
            "epoch": epoch,
            **train_metrics,
            "val_loss": val_metrics["loss"],
            "val_accuracy": val_metrics["accuracy"],
            "val_macro_precision": val_metrics["macro_precision"],
            "val_macro_recall": val_metrics["macro_recall"],
            "val_macro_f1": val_metrics["macro_f1"]
        }

        history.append(epoch_record)

        pd.DataFrame(history).to_csv(
            log_path, index=False
        )

        print(
            f"Train loss: {train_metrics['train_loss']:.4f} | "
            f"Validation loss: {val_metrics['loss']:.4f}"
        )

        print(
            f"Validation accuracy: "
            f"{val_metrics['accuracy']:.4f} | "
            f"Macro F1: {val_metrics['macro_f1']:.4f}"
        )

        # Save best model based on validation macro-F1
        if val_metrics["macro_f1"] > best_f1:

            best_f1 = val_metrics["macro_f1"]
            best_epoch = epoch
            patience_counter = 0

            checkpoint = {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "criterion_state_dict": criterion.state_dict(),
                "best_validation_macro_f1": best_f1,
                "config": CONFIG,
                "loss_type": loss_type
            }

            torch.save(
                checkpoint,
                os.path.join(
                    checkpoint_dir,
                    "best_model.pth"
                )
            )

            print("Best checkpoint saved.")

        else:

            patience_counter += 1

        # Early stopping
        if patience_counter >= CONFIG["early_stopping_patience"]:

            print("Early stopping triggered.")
            break

    print("\nEXPERIMENT FINISHED")
    print("Best epoch:", best_epoch)
    print("Best validation macro-F1:", best_f1)
    print("Log saved:", log_path)

    return {
        "model": model,
        "criterion": criterion,
        "history": pd.DataFrame(history),
        "best_epoch": best_epoch,
        "best_validation_macro_f1": best_f1,
        "checkpoint_dir": checkpoint_dir
    }


print("TRAINING ENGINE CREATED SUCCESSFULLY")

In [ ]:
import inspect

required_functions = [
    train_one_epoch,
    evaluate_model,
    update_acal_statistics,
    create_experiment_train_loader,
    run_experiment
]

for function in required_functions:
    print(
        function.__name__,
        "->",
        "READY" if callable(function) else "NOT READY"
    )

print("\nCUDA device:", torch.cuda.get_device_name(0))
print("Mixed precision support: configured")
print("Training epochs configured:", CONFIG["epochs"])
print("\nTRAINING ENGINE VERIFICATION COMPLETED")

In [ ]:
import os
import torch

# --------------------------------------------------
# 1. CREATE SHARED INITIAL MODEL WEIGHTS
# --------------------------------------------------

set_seed(SEED)

initial_model = PredatorNet(
    num_classes=NUM_CLASSES,
    pretrained=False
).to(DEVICE)

# Keep an independent CPU copy
INITIAL_MODEL_STATE = {
    key: value.detach().cpu().clone()
    for key, value in initial_model.state_dict().items()
}

INITIAL_WEIGHTS_PATH = os.path.join(
    BASE_OUTPUT,
    "shared_initial_predatornet.pth"
)

torch.save(
    {
        "model_state_dict": INITIAL_MODEL_STATE,
        "seed": SEED,
        "architecture": "PredatorNet"
    },
    INITIAL_WEIGHTS_PATH
)

print("Shared initial weights saved:", INITIAL_WEIGHTS_PATH)


# --------------------------------------------------
# 2. TEST BOTH LOSS FUNCTIONS
# --------------------------------------------------

initial_model.eval()

smoke_input = torch.randn(
    2, 3, IMAGE_SIZE, IMAGE_SIZE,
    device=DEVICE
)

smoke_targets = torch.tensor(
    [0, 2],
    dtype=torch.long,
    device=DEVICE
)

# Test BGRA
bgra_criterion = BaselineBGRALoss(
    lambda_smooth=CONFIG["lambda_smooth"]
).to(DEVICE)

initial_model.zero_grad(set_to_none=True)

bgra_output = initial_model(smoke_input)
bgra_result = bgra_criterion(bgra_output, smoke_targets)

bgra_result["total_loss"].backward()

bgra_gradient_ok = all(
    p.grad is None or torch.isfinite(p.grad).all().item()
    for p in initial_model.parameters()
)

print("\nBGRA SMOKE TEST")
print("Loss:", bgra_result["total_loss"].item())
print("Finite gradients:", bgra_gradient_ok)


# Test ACAL
acal_criterion = AdaptiveConfusionAwareLoss(
    lambda_smooth=CONFIG["lambda_smooth"],
    lambda_confusion=CONFIG["lambda_confusion"],
    num_classes=NUM_CLASSES,
    ema_momentum=CONFIG["ema_momentum"],
    smoothing=CONFIG["confusion_smoothing"]
).to(DEVICE)

initial_model.zero_grad(set_to_none=True)

acal_output = initial_model(smoke_input)
acal_result = acal_criterion(acal_output, smoke_targets)

acal_result["total_loss"].backward()

acal_gradient_ok = all(
    p.grad is None or torch.isfinite(p.grad).all().item()
    for p in initial_model.parameters()
)

# Test the ACAL statistics mechanism on dummy data only
acal_criterion.update_training_statistics(
    acal_output["logits"],
    smoke_targets
)

acal_criterion.finish_training_epoch()

print("\nACAL SMOKE TEST")
print("Loss:", acal_result["total_loss"].item())
print("Finite gradients:", acal_gradient_ok)
print("Confusion weights updated:", torch.isfinite(
    acal_criterion.confusion_weights
).all().item())

# Clear temporary gradients and GPU memory
initial_model.zero_grad(set_to_none=True)
del initial_model, bgra_output, acal_output
torch.cuda.empty_cache()

print("\nSHARED INITIAL WEIGHTS:", os.path.exists(INITIAL_WEIGHTS_PATH))
print("BOTH LOSS TESTS PASSED:", bgra_gradient_ok and acal_gradient_ok)
print("\nSMOKE TEST COMPLETED — NO TRAINING PERFORMED")

In [ ]:
# Load the shared initial weights
initial_checkpoint = torch.load(
    INITIAL_WEIGHTS_PATH,
    map_location="cpu",
    weights_only=False
)

shared_initial_state = initial_checkpoint["model_state_dict"]

# Verify architecture initialization
verification_model = PredatorNet(
    num_classes=NUM_CLASSES,
    pretrained=False
)

verification_model.load_state_dict(shared_initial_state)

print("Shared initial weights verified.")

del verification_model

# Start baseline BGRA training
baseline_results = run_experiment(
    experiment_name="baseline",
    loss_type="BGRA",
    initial_state=shared_initial_state,
    epochs=CONFIG["epochs"]
)

print("\nBASELINE TRAINING COMPLETED")
print("Best epoch:", baseline_results["best_epoch"])
print(
    "Best validation macro-F1:",
    baseline_results["best_validation_macro_f1"]
)

In [ ]:
# Load the SAME initial weights used by baseline
initial_checkpoint = torch.load(
    INITIAL_WEIGHTS_PATH,
    map_location="cpu",
    weights_only=False
)

shared_initial_state = initial_checkpoint["model_state_dict"]

# Start proposed ACAL experiment
acal_results = run_experiment(
    experiment_name="acal",
    loss_type="ACAL",
    initial_state=shared_initial_state,
    epochs=CONFIG["epochs"]
)

print("\nACAL TRAINING COMPLETED")

print("Best epoch:", acal_results["best_epoch"])

print(
    "Best validation macro-F1:",
    acal_results["best_validation_macro_f1"]
)

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

RESULT_DIR = "/kaggle/working/acal_ad/training/comparison"
os.makedirs(RESULT_DIR, exist_ok=True)

# Read actual training logs
bgra_log = pd.read_csv(
    "/kaggle/working/acal_ad/training/baseline/training_log.csv"
)

acal_log = pd.read_csv(
    "/kaggle/working/acal_ad/training/acal/training_log.csv"
)

# Save comparison summary
comparison = pd.DataFrame([
    {
        "Experiment": "Baseline BGRA",
        "Best Epoch": 24,
        "Best Validation Macro F1": 0.9622303745297149,
        "Validation Accuracy at Best Epoch": 0.9479,
        "Validation Loss at Best Epoch": 0.1549
    },
    {
        "Experiment": "Proposed ACAL",
        "Best Epoch": 19,
        "Best Validation Macro F1": 0.9626643021225227,
        "Validation Accuracy at Best Epoch": 0.9510,
        "Validation Loss at Best Epoch": 0.1403
    }
])

comparison.to_csv(
    os.path.join(RESULT_DIR, "experiment_comparison.csv"),
    index=False
)

# Plot validation macro-F1
plt.figure(figsize=(10, 5))

plt.plot(
    bgra_log["epoch"],
    bgra_log["val_macro_f1"],
    marker="o",
    label="Baseline BGRA"
)

plt.plot(
    acal_log["epoch"],
    acal_log["val_macro_f1"],
    marker="s",
    label="Proposed ACAL"
)

plt.xlabel("Epoch")
plt.ylabel("Validation Macro-F1")
plt.title("Baseline BGRA vs Proposed ACAL")
plt.legend()
plt.grid(True)

plt.savefig(
    os.path.join(RESULT_DIR, "validation_macro_f1.png"),
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("COMPARISON FILES SAVED")
print(comparison)
print("\nSaved in:", RESULT_DIR)

In [ ]:
import os
import shutil
import zipfile

BASE = "/kaggle/working/acal_ad"

BACKUP_DIR = os.path.join(
    BASE, "backup_70_15_15"
)

os.makedirs(BACKUP_DIR, exist_ok=True)

# 1. Back up the existing dataset package
source_dataset = os.path.join(
    BASE, "final_dataset_package"
)

if os.path.exists(source_dataset):
    destination = os.path.join(
        BACKUP_DIR, "dataset_70_15_15"
    )

    if not os.path.exists(destination):
        shutil.copytree(source_dataset, destination)

# 2. Back up completed training experiments
source_training = os.path.join(
    BASE, "training"
)

destination_training = os.path.join(
    BACKUP_DIR, "training_70_15_15"
)

if os.path.exists(source_training):
    if not os.path.exists(destination_training):
        shutil.copytree(source_training, destination_training)

# 3. Back up current manifest
if "original_manifest" in globals():
    original_manifest.to_csv(
        os.path.join(
            BACKUP_DIR,
            "manifest_70_15_15.csv"
        ),
        index=False
    )

# 4. Create ZIP backup
zip_path = shutil.make_archive(
    "/kaggle/working/ACAL_AD_Experiment1_70_15_15",
    "zip",
    BACKUP_DIR
)

print("EXPERIMENT 1 BACKUP COMPLETED")
print("--------------------------------")
print("Backup directory:", BACKUP_DIR)
print("ZIP file:", zip_path)

print("\nBackup contents:")
for item in os.listdir(BACKUP_DIR):
    print("-", item)

In [ ]:
import os
from collections import Counter

DATASET_PATH = (
    "/kaggle/input/datasets/"
    "uraninjo/augmented-alzheimer-mri-dataset"
)

ORIGINAL_PATH = os.path.join(
    DATASET_PATH, "OriginalDataset"
)

AUGMENTED_PATH = os.path.join(
    DATASET_PATH, "AugmentedAlzheimerDataset"
)

CLASSES = [
    "MildDemented",
    "ModerateDemented",
    "NonDemented",
    "VeryMildDemented"
]


def inspect_folder(folder_path):
    results = {}

    for class_name in CLASSES:
        class_path = os.path.join(
            folder_path, class_name
        )

        files = [
            f for f in os.listdir(class_path)
            if f.lower().endswith((".jpg", ".jpeg", ".png"))
        ]

        results[class_name] = {
            "count": len(files),
            "examples": files[:10]
        }

    return results


original_info = inspect_folder(ORIGINAL_PATH)
augmented_info = inspect_folder(AUGMENTED_PATH)

print("ORIGINAL DATASET")
print("----------------")

for class_name, info in original_info.items():
    print(f"\n{class_name}: {info['count']}")
    print("Examples:", info["examples"])


print("\n\nAUGMENTED DATASET")
print("-----------------")

for class_name, info in augmented_info.items():
    print(f"\n{class_name}: {info['count']}")
    print("Examples:", info["examples"])


print("\nTOTAL ORIGINAL:",
      sum(x["count"] for x in original_info.values()))

print("TOTAL AUGMENTED:",
      sum(x["count"] for x in augmented_info.values()))

In [ ]:

import os
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

# Use the existing original-only manifest
df = original_manifest.copy().reset_index(drop=True)

# Verify required columns
required = ["class", "sha256", "kaggle_path"]
for col in required:
    assert col in df.columns, f"Missing column: {col}"

assert len(df) == 6400, f"Expected 6400 images, found {len(df)}"

# Group exact duplicates using SHA-256
X = df[["kaggle_path"]]
y = df["class"]
groups = df["sha256"]

# First split: approximately 80% training and 20% temporary
splitter_80 = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

train_idx, temp_idx = next(splitter_80.split(X, y, groups))

df["split"] = "train"

# Divide remaining 20% into validation and test
temp_df = df.iloc[temp_idx].copy()
temp_X = temp_df[["kaggle_path"]]
temp_y = temp_df["class"]
temp_groups = temp_df["sha256"]

splitter_10 = StratifiedGroupKFold(
    n_splits=2,
    shuffle=True,
    random_state=42
)

val_local_idx, test_local_idx = next(
    splitter_10.split(temp_X, temp_y, temp_groups)
)

val_idx = temp_df.index[val_local_idx]
test_idx = temp_df.index[test_local_idx]

df.loc[val_idx, "split"] = "validation"
df.loc[test_idx, "split"] = "test"

# Save separately; do not overwrite the old manifest
OUTPUT_PATH = "/kaggle/working/acal_ad/original_only_manifest_80_10_10.csv"

os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
df.to_csv(OUTPUT_PATH, index=False)

# Display split counts
print("NEW 80:10:10 SPLIT")
print("-------------------")
print(df["split"].value_counts())

print("\nCLASS-WISE DISTRIBUTION")
print(pd.crosstab(df["class"], df["split"]))

# Verify exact duplicate groups do not cross splits
duplicate_check = df.groupby("sha256")["split"].nunique()
cross_split_duplicates = (duplicate_check > 1).sum()

print("\nEXACT DUPLICATE GROUPS CROSSING SPLITS:", cross_split_duplicates)
print("\nMANIFEST SAVED:", OUTPUT_PATH)


In [ ]:
print("DATASET CLASS TEST")

sample_dataset = AlzheimerDataset(
    df[df["split"] == "train"].head(2),
    transform=None
)

print("Dataset length:", len(sample_dataset))

sample = sample_dataset[0]

print("Returned object type:", type(sample))

if isinstance(sample, (tuple, list)):
    print("Number of returned elements:", len(sample))

    for i, item in enumerate(sample):
        if hasattr(item, "shape"):
            print(f"Element {i} shape:", item.shape)
            print(f"Element {i} type:", item.dtype)
        else:
            print(f"Element {i} value:", item)
else:
    print("Returned object:", sample)

In [ ]:
print("CLASS MAPPING")
print(CLASS_NAMES)

print("\nEXISTING TRANSFORMATIONS")

print("Training transform:")
print(train_dataset.transform)

print("\nValidation transform:")
print(val_dataset.transform)

print("\nTEST TRANSFORM:")
print(test_dataset.transform)

print("\nEXISTING DATASET SIZES")
print("Training:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Testing:", len(test_dataset))

In [ ]:

import os
import numpy as np
import torch
from torch.utils.data import DataLoader, WeightedRandomSampler

# Load the new manifest
MANIFEST_80_PATH = (
    "/kaggle/working/acal_ad/"
    "original_only_manifest_80_10_10.csv"
)

manifest_80 = pd.read_csv(MANIFEST_80_PATH)

# Separate the three splits
train_df_80 = manifest_80[
    manifest_80["split"] == "train"
].reset_index(drop=True)

val_df_80 = manifest_80[
    manifest_80["split"] == "validation"
].reset_index(drop=True)

test_df_80 = manifest_80[
    manifest_80["split"] == "test"
].reset_index(drop=True)

# Reuse the existing transformations
train_dataset_80 = AlzheimerDataset(
    train_df_80,
    transform=train_dataset.transform
)

val_dataset_80 = AlzheimerDataset(
    val_df_80,
    transform=val_dataset.transform
)

test_dataset_80 = AlzheimerDataset(
    test_df_80,
    transform=test_dataset.transform
)

# Separate loader for ACAL training statistics
train_stats_dataset_80 = AlzheimerDataset(
    train_df_80,
    transform=train_dataset.transform
)

# Calculate class-balanced sampling weights
train_labels_80 = np.array([
    CLASS_NAMES.index(class_name)
    for class_name in train_df_80["class"]
])

class_counts_80 = np.bincount(
    train_labels_80,
    minlength=len(CLASS_NAMES)
)

class_weights_80 = 1.0 / class_counts_80
sample_weights_80 = class_weights_80[train_labels_80]

sampler_generator_80 = torch.Generator()
sampler_generator_80.manual_seed(42)

train_sampler_80 = WeightedRandomSampler(
    weights=torch.as_tensor(sample_weights_80, dtype=torch.double),
    num_samples=len(train_dataset_80),
    replacement=True,
    generator=sampler_generator_80
)

# Create DataLoaders
train_loader_80 = DataLoader(
    train_dataset_80,
    batch_size=BATCH_SIZE,
    sampler=train_sampler_80,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

train_stats_loader_80 = DataLoader(
    train_stats_dataset_80,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

val_loader_80 = DataLoader(
    val_dataset_80,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

test_loader_80 = DataLoader(
    test_dataset_80,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

# Print verification
print("80:10:10 DATASET AND DATALOADER VERIFICATION")
print("--------------------------------------------")

print("Training images:", len(train_dataset_80))
print("Validation images:", len(val_dataset_80))
print("Testing images:", len(test_dataset_80))

print("\nTraining batches:", len(train_loader_80))
print("Training statistics batches:", len(train_stats_loader_80))
print("Validation batches:", len(val_loader_80))
print("Testing batches:", len(test_loader_80))

print("\nTraining class counts:")
for i, name in enumerate(CLASS_NAMES):
    print(name, ":", class_counts_80[i])

print("\nAll four DataLoaders created successfully.")


In [ ]:
# Get one batch from each loader
images, labels = next(iter(train_loader_80))
val_images, val_labels = next(iter(val_loader_80))
test_images, test_labels = next(iter(test_loader_80))

print("BATCH VERIFICATION")
print("------------------")

print("\nTRAINING BATCH")
print("Image shape:", images.shape)
print("Label shape:", labels.shape)
print("Image dtype:", images.dtype)
print("Label dtype:", labels.dtype)
print("Image range:", images.min().item(), images.max().item())

print("\nVALIDATION BATCH")
print("Image shape:", val_images.shape)
print("Label shape:", val_labels.shape)

print("\nTEST BATCH")
print("Image shape:", test_images.shape)
print("Label shape:", test_labels.shape)

print("\nLABEL MAPPING")
for i, name in enumerate(CLASS_NAMES):
    print(i, "=", name)

print("\nBATCH VERIFICATION COMPLETED")

In [ ]:
import inspect

print("TRAINING FUNCTION SIGNATURES")
print("-----------------------------")

for name in [
    "run_experiment",
    "train_one_epoch",
    "evaluate_model",
    "update_acal_statistics",
    "create_experiment_train_loader"
]:
    fn = globals().get(name)

    if fn is not None:
        print(f"\n{name}{inspect.signature(fn)}")
        print("Global variables used:", fn.__code__.co_names)
    else:
        print(f"\n{name}: NOT FOUND")

In [ ]:
print("TRAINING CONFIGURATION")
print("----------------------")

for key, value in CONFIG.items():
    print(f"{key}: {value}")

print("\nEXISTING OUTPUT DIRECTORY")
print("BASE_OUTPUT:", BASE_OUTPUT)

print("\nSHARED INITIAL WEIGHTS")
print("File exists:", os.path.exists(
    "/kaggle/working/acal_ad/training/shared_initial_predatornet.pth"
))

print("\nOLD DATASET VARIABLES")
print("train_df:", len(train_df))
print("train_dataset:", len(train_dataset))
print("train_stats_loader:", len(train_stats_loader))
print("val_loader:", len(val_loader))

print("\nNEW DATASET VARIABLES")
print("train_df_80:", len(train_df_80))
print("train_dataset_80:", len(train_dataset_80))
print("train_stats_loader_80:", len(train_stats_loader_80))
print("val_loader_80:", len(val_loader_80))

In [ ]:

import os
import torch
from torch.utils.data import DataLoader, WeightedRandomSampler

# 1. Create a separate output directory
BASE_OUTPUT_80 = "/kaggle/working/acal_ad/training_80_10_10"
os.makedirs(BASE_OUTPUT_80, exist_ok=True)

# 2. Redirect the existing training engine to the new loaders
train_stats_loader = train_stats_loader_80
val_loader = val_loader_80

# 3. Create a fresh, reproducible training sampler for each experiment
def create_experiment_train_loader(seed=42):
    generator = torch.Generator()
    generator.manual_seed(seed)

    sampler = WeightedRandomSampler(
        weights=torch.as_tensor(
            sample_weights_80,
            dtype=torch.double
        ),
        num_samples=len(train_dataset_80),
        replacement=True,
        generator=generator
    )

    return DataLoader(
        train_dataset_80,
        batch_size=BATCH_SIZE,
        sampler=sampler,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available()
    )

# 4. Redirect output location
BASE_OUTPUT = BASE_OUTPUT_80

# 5. Load the shared initial model weights
INITIAL_WEIGHTS_PATH = (
    "/kaggle/working/acal_ad/training/"
    "shared_initial_predatornet.pth"
)

assert os.path.exists(INITIAL_WEIGHTS_PATH), "Initial weights not found!"

shared_initial_state_80 = torch.load(
    INITIAL_WEIGHTS_PATH,
    map_location="cpu",
    weights_only=True
)

# 6. Verify everything
print("80:10:10 TRAINING ENGINE PREPARATION")
print("------------------------------------")

print("Training dataset:", len(train_dataset_80))
print("Statistics loader batches:", len(train_stats_loader))
print("Validation loader batches:", len(val_loader))

print("Output directory:", BASE_OUTPUT)
print("Initial weights loaded:", isinstance(shared_initial_state_80, dict))
print("Initial weight entries:", len(shared_initial_state_80))

print("Fresh sampler batches:", len(create_experiment_train_loader(42)))

print("\nPrevious experiment output remains separate:")
print("/kaggle/working/acal_ad/training")

print("\n80:10:10 training engine is prepared.")


In [ ]:
print("SHARED INITIAL WEIGHTS STRUCTURE")
print("--------------------------------")

print("Top-level keys:")
for key, value in shared_initial_state_80.items():
    print("\nKey:", key)
    print("Value type:", type(value))

    if isinstance(value, dict):
        print("Nested entries:", len(value))
        print("First 10 nested keys:", list(value.keys())[:10])
    elif hasattr(value, "shape"):
        print("Tensor shape:", value.shape)
    else:
        print("Value:", value if isinstance(value, (str, int, float)) else "Other object")

print("\nPREDATORNET MODEL STATE")
temp_model = PredatorNet(NUM_CLASSES)
model_keys = temp_model.state_dict()

print("Model parameter entries:", len(model_keys))
print("First 10 model keys:", list(model_keys.keys())[:10])

In [ ]:
# Extract the actual model weights
initial_weights_80 = shared_initial_state_80["model_state_dict"]

# Verify exact model-key compatibility
model_keys = set(temp_model.state_dict().keys())
checkpoint_keys = set(initial_weights_80.keys())

print("CHECKPOINT COMPATIBILITY")
print("------------------------")
print("Matching keys:", len(model_keys & checkpoint_keys))
print("Missing keys:", len(model_keys - checkpoint_keys))
print("Unexpected keys:", len(checkpoint_keys - model_keys))

# Verify that weights load correctly
temp_model.load_state_dict(initial_weights_80, strict=True)

print("\nStrict model loading: SUCCESS")

# Inspect the accepted loss-type conditions
print("\nTRAINING FUNCTION LOSS OPTIONS")
print("------------------------------")
print("Function constants:")
print(run_experiment.__code__.co_consts)

# Verify output directory
print("\n80:10:10 output directory:", BASE_OUTPUT)

In [ ]:

# EXPERIMENT 1: PREDATORNET + BGRA
# Dataset split: 80:10:10
# Training images: 5120

print("STARTING 80:10:10 BASELINE EXPERIMENT")
print("=====================================")

baseline_80_results = run_experiment(
    experiment_name="baseline_bgra_80_10_10",
    loss_type="BGRA",
    initial_state=initial_weights_80,
    epochs=30
)

print("\nBASELINE EXPERIMENT COMPLETED")
print("Best epoch:", baseline_80_results["best_epoch"])
print(
    "Best validation macro-F1:",
    baseline_80_results["best_validation_macro_f1"]
)


In [ ]:

# EXPERIMENT 2: PREDATORNET + ACAL
# Dataset split: 80:10:10

print("STARTING 80:10:10 ACAL EXPERIMENT")
print("=================================")

acal_80_results = run_experiment(
    experiment_name="acal_80_10_10",
    loss_type="ACAL",
    initial_state=initial_weights_80,
    epochs=30
)

print("\nACAL EXPERIMENT COMPLETED")
print("Best epoch:", acal_80_results["best_epoch"])
print(
    "Best validation macro-F1:",
    acal_80_results["best_validation_macro_f1"]
)


In [ ]:
# SWITCH EXPERIMENT CONFIGURATION TO SEED 123

SEED = 123
CONFIG["seed"] = 123

BASE_OUTPUT_123 = (
    "/kaggle/working/acal_ad/training_80_10_10_seed123"
)

BASE_OUTPUT = BASE_OUTPUT_123

os.makedirs(BASE_OUTPUT, exist_ok=True)

print("SEED:", SEED)
print("CONFIG SEED:", CONFIG["seed"])
print("OUTPUT DIRECTORY:", BASE_OUTPUT)
print("TRAINING DATASET:", len(train_dataset_80))
print("VALIDATION DATASET:", len(val_dataset_80))
print("TEST DATASET:", len(test_dataset_80))

In [ ]:
import os

seed123_bgra_checkpoint = (
    "/kaggle/working/acal_ad/training_80_10_10_seed123/"
    "baseline_bgra_seed123/checkpoints/best_model.pth"
)

print("SEED-123 BGRA CHECKPOINT:")
print(seed123_bgra_checkpoint)

print("\nCHECKPOINT EXISTS:",
      os.path.exists(seed123_bgra_checkpoint))

if os.path.exists(seed123_bgra_checkpoint):
    print("FILE SIZE (MB):",
          round(os.path.getsize(seed123_bgra_checkpoint) / (1024 * 1024), 2))

In [ ]:
import os

search_root = "/kaggle/working/acal_ad"

print("SEARCHING FOR SAVED SEED 123 MODELS")
print("===================================")

for root, dirs, files in os.walk(search_root):
    for file in files:
        if file.endswith((".pth", ".pt")):
            full_path = os.path.join(root, file)
            print(full_path)

In [ ]:
import os
import torch

initial_checkpoint_path = (
    "/kaggle/working/acal_ad/training/"
    "shared_initial_predatornet.pth"
)

print("Shared initial checkpoint exists:",
      os.path.exists(initial_checkpoint_path))

if os.path.exists(initial_checkpoint_path):
    initial_checkpoint = torch.load(
        initial_checkpoint_path,
        map_location="cpu",
        weights_only=False
    )

    print("Checkpoint keys:", initial_checkpoint.keys())
    print("Saved seed:", initial_checkpoint.get("seed"))
    print("Architecture:", initial_checkpoint.get("architecture"))
    print("Model parameter entries:",
          len(initial_checkpoint["model_state_dict"]))

In [ ]:
import torch
import os

initial_checkpoint_path = (
    "/kaggle/working/acal_ad/training/"
    "shared_initial_predatornet.pth"
)

initial_checkpoint = torch.load(
    initial_checkpoint_path,
    map_location="cpu",
    weights_only=False
)

initial_weights_80 = initial_checkpoint["model_state_dict"]

print("INITIAL WEIGHTS LOADED")
print("======================")
print("Architecture:", initial_checkpoint["architecture"])
print("Initialization seed:", initial_checkpoint["seed"])
print("Parameter entries:", len(initial_weights_80))
print("Experiment seed:", SEED)
print("Output directory:", BASE_OUTPUT)

In [ ]:
# EXPERIMENT 1: PREDATORNET + BGRA
# Dataset split: 80:10:10
# Experiment seed: 123

print("STARTING SEED-123 BGRA EXPERIMENT")
print("==================================")

bgra_123_results = run_experiment(
    experiment_name="baseline_bgra_seed123",
    loss_type="BGRA",
    initial_state=initial_weights_80,
    epochs=30
)

print("\nSEED-123 BGRA EXPERIMENT COMPLETED")
print("Best epoch:", bgra_123_results["best_epoch"])
print(
    "Best validation macro-F1:",
    bgra_123_results["best_validation_macro_f1"]
)

In [ ]:
import os

bgra_123_checkpoint = (
    "/kaggle/working/acal_ad/training_80_10_10_seed123/"
    "baseline_bgra_seed123/checkpoints/best_model.pth"
)

print("SEED 123 BGRA CHECKPOINT")
print("========================")
print("Path:", bgra_123_checkpoint)
print("Exists:", os.path.exists(bgra_123_checkpoint))

if os.path.exists(bgra_123_checkpoint):
    print(
        "File size (MB):",
        round(os.path.getsize(bgra_123_checkpoint) / (1024 * 1024), 2)
    )

In [ ]:
# EXPERIMENT 2: PREDATORNET + ACAL
# Dataset split: 80:10:10
# Experiment seed: 123

print("STARTING SEED-123 ACAL EXPERIMENT")
print("=================================")

acal_123_results = run_experiment(
    experiment_name="acal_seed123",
    loss_type="ACAL",
    initial_state=initial_weights_80,
    epochs=30
)

print("\nSEED-123 ACAL EXPERIMENT COMPLETED")
print("Best epoch:", acal_123_results["best_epoch"])
print(
    "Best validation macro-F1:",
    acal_123_results["best_validation_macro_f1"]
)

In [ ]:
import os

acal_123_checkpoint = (
    "/kaggle/working/acal_ad/training_80_10_10_seed123/"
    "acal_seed123/checkpoints/best_model.pth"
)

print("SEED 123 ACAL CHECKPOINT")
print("========================")
print("Path:", acal_123_checkpoint)
print("Exists:", os.path.exists(acal_123_checkpoint))

if os.path.exists(acal_123_checkpoint):
    print(
        "File size (MB):",
        round(os.path.getsize(acal_123_checkpoint) / (1024 * 1024), 2)
    )

In [ ]:
import os
import torch

model_paths = {
    "BGRA_Seed42": (
        "/kaggle/working/acal_ad/training_80_10_10/"
        "baseline_bgra_80_10_10/checkpoints/best_model.pth"
    ),
    "ACAL_Seed42": (
        "/kaggle/working/acal_ad/training_80_10_10/"
        "acal_80_10_10/checkpoints/best_model.pth"
    ),
    "BGRA_Seed123": (
        "/kaggle/working/acal_ad/training_80_10_10_seed123/"
        "baseline_bgra_seed123/checkpoints/best_model.pth"
    ),
    "ACAL_Seed123": (
        "/kaggle/working/acal_ad/training_80_10_10_seed123/"
        "acal_seed123/checkpoints/best_model.pth"
    )
}

print("CHECKING ALL FOUR EXPERIMENT CHECKPOINTS")
print("========================================")

for name, path in model_paths.items():
    print(f"\n{name}")
    print("Exists:", os.path.exists(path))

    if os.path.exists(path):
        checkpoint = torch.load(
            path,
            map_location="cpu",
            weights_only=False
        )
        print("Checkpoint keys:", list(checkpoint.keys()))
        print("File size (MB):",
              round(os.path.getsize(path) / (1024 * 1024), 2))

In [ ]:
import os
import torch
import json
import shutil

# Export directory
EXPORT_DIR = "/kaggle/working/ACAL_AD_Final_Models"
os.makedirs(EXPORT_DIR, exist_ok=True)

# Four trained checkpoints
model_paths = {
    "PredatorNet_BGRA_Seed42.pt": (
        "/kaggle/working/acal_ad/training_80_10_10/"
        "baseline_bgra_80_10_10/checkpoints/best_model.pth"
    ),
    "PredatorNet_ACAL_Seed42.pt": (
        "/kaggle/working/acal_ad/training_80_10_10/"
        "acal_80_10_10/checkpoints/best_model.pth"
    ),
    "PredatorNet_BGRA_Seed123.pt": (
        "/kaggle/working/acal_ad/training_80_10_10_seed123/"
        "baseline_bgra_seed123/checkpoints/best_model.pth"
    ),
    "PredatorNet_ACAL_Seed123.pt": (
        "/kaggle/working/acal_ad/training_80_10_10_seed123/"
        "acal_seed123/checkpoints/best_model.pth"
    )
}

# Export checkpoint files
for filename, source_path in model_paths.items():

    if not os.path.exists(source_path):
        raise FileNotFoundError(source_path)

    checkpoint = torch.load(
        source_path,
        map_location="cpu",
        weights_only=False
    )

    output_path = os.path.join(EXPORT_DIR, filename)

    torch.save({
        "model_state_dict": checkpoint["model_state_dict"],
        "epoch": checkpoint["epoch"],
        "best_validation_macro_f1":
            checkpoint["best_validation_macro_f1"],
        "config": checkpoint["config"],
        "loss_type": checkpoint["loss_type"],
        "architecture": "PredatorNet",
        "dataset_split": "80:10:10",
        "class_names": [
            "MildDemented",
            "ModerateDemented",
            "NonDemented",
            "VeryMildDemented"
        ],
        "input_size": 224,
        "normalization_mean": [0.5, 0.5, 0.5],
        "normalization_std": [0.5, 0.5, 0.5]
    }, output_path)

    print("SAVED:", filename)
    print("SIZE (MB):",
          round(os.path.getsize(output_path) / (1024 * 1024), 2))
    print("--------------------------------")

print("\nALL FOUR MODELS EXPORTED SUCCESSFULLY")
print("EXPORT DIRECTORY:", EXPORT_DIR)